# Load Pretrained EvoIR Model from HuggingFace

Downloads and maps pretrained weights from `leonmakise/EvoIR` to our AdaIR architecture.

---

In [ ]:
import torch
import os
from pathlib import Path

print(f'PyTorch: {torch.__version__}')

## 1. Download from HuggingFace

In [ ]:
def download_evoIR_weights(
    repo_id='leonmakise/EvoIR',
    save_dir='./pretrained',
    filename=None
):
    """
    Download EvoIR pretrained weights from HuggingFace Hub.
    
    Args:
        repo_id: HuggingFace repo ID
        save_dir: Local directory to save weights
        filename: Specific file to download (None = all .pth files)
    Returns:
        Path to downloaded checkpoint
    """
    os.makedirs(save_dir, exist_ok=True)
    
    try:
        from huggingface_hub import hf_hub_download, list_repo_files
        
        # List available files
        files = list_repo_files(repo_id)
        model_files = [f for f in files if f.endswith(('.pth', '.pt', '.ckpt', '.safetensors'))]
        
        if not model_files:
            print(f'No model files found in {repo_id}')
            print(f'Available files: {files}')
            return None
        
        print(f'Found model files: {model_files}')
        
        # Download specified or first available
        target = filename if filename else model_files[0]
        local_path = hf_hub_download(
            repo_id=repo_id,
            filename=target,
            local_dir=save_dir
        )
        
        print(f'Downloaded: {local_path}')
        return local_path
        
    except ImportError:
        print('huggingface_hub not installed. Install with: pip install huggingface_hub')
        print(f'\nManual download: https://huggingface.co/{repo_id}/tree/main')
        return None
    except Exception as e:
        print(f'Download failed: {e}')
        print(f'\nManual download: https://huggingface.co/{repo_id}/tree/main')
        return None

print('download_evoIR_weights defined')

## 2. Load and Map Weights

In [ ]:
def load_pretrained_weights(model, checkpoint_path, strict=False):
    """
    Load pretrained weights into model, handling key mismatches gracefully.
    
    Args:
        model: Our AdaIR model instance
        checkpoint_path: Path to .pth checkpoint
        strict: If True, raise error on key mismatch
    Returns:
        Tuple of (matched_keys, missing_keys, unexpected_keys)
    """
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')
    
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    
    # Handle different checkpoint formats
    if isinstance(checkpoint, dict):
        if 'model_state_dict' in checkpoint:
            state_dict = checkpoint['model_state_dict']
        elif 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
        elif 'model' in checkpoint:
            state_dict = checkpoint['model']
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint
    
    # Remove 'module.' prefix from DDP training
    cleaned = {}
    for k, v in state_dict.items():
        new_key = k.replace('module.', '')
        cleaned[new_key] = v
    
    # Map EvoIR keys to our naming convention
    key_map = {
        'patch_embed.proj': 'pe',
        'encoder_level1.': 'e1.',
        'encoder_level1_fft.': 'e1f.',
        'encoder_level2.': 'e2.',
        'encoder_level2_fft.': 'e2f.',
        'encoder_level3.': 'e3.',
        'encoder_level3_fft.': 'e3f.',
        'down1_2': 'd1',
        'down2_3': 'd2',
        'down3_4': 'd3',
        'latent.': 'lat.',
        'latent_fft.': 'latf.',
        'up4_3': 'u3',
        'up3_2': 'u2',
        'up2_1': 'u1',
        'reduce_chan_level3': 'r3',
        'reduce_chan_level2': 'r2',
        'reduce_chan_level1': 'r1',
        'decoder_level3.': 'dc3.',
        'decoder_level3_fft.': 'dc3f.',
        'decoder_level2.': 'dc2.',
        'decoder_level2_fft.': 'dc2f.',
        'decoder_level1.': 'dc1.',
        'decoder_level1_fft.': 'dc1f.',
        'refinement.': 'refine.',
    }
    
    mapped = {}
    for k, v in cleaned.items():
        new_key = k
        for old, new in key_map.items():
            new_key = new_key.replace(old, new)
        mapped[new_key] = v
    
    # Try loading
    result = model.load_state_dict(mapped, strict=strict)
    
    matched = len(model.state_dict()) - len(result.missing_keys)
    print(f'Loaded weights:')
    print(f'  Matched: {matched}')
    print(f'  Missing: {len(result.missing_keys)}')
    print(f'  Unexpected: {len(result.unexpected_keys)}')
    
    if result.missing_keys:
        print(f'\nMissing keys (first 10): {result.missing_keys[:10]}')
    if result.unexpected_keys:
        print(f'\nUnexpected keys (first 10): {result.unexpected_keys[:10]}')
    
    return matched, result.missing_keys, result.unexpected_keys

print('load_pretrained_weights defined')

## 3. Usage Example

In [ ]:
# Download weights
print('=== Downloading EvoIR weights ===')
ckpt_path = download_evoIR_weights()

# If download successful, load into model
if ckpt_path and os.path.exists(ckpt_path):
    # Import model from train or test notebook
    # For standalone use, define AdaIR model here or import
    print(f'\nCheckpoint ready at: {ckpt_path}')
    print('\nTo load into model:')
    print('  model = AdaIR()')
    print(f'  load_pretrained_weights(model, "{ckpt_path}")')
else:
    print('\nNo pretrained weights available.')
    print('The model will train from scratch using train.ipynb.')
    print('\nAlternatively, download manually:')
    print('  https://huggingface.co/leonmakise/EvoIR/tree/main')